# 🎯 Objetivos

El objetivo de este apartado es construir un modelo de recomendación base por popularidad que sirva como punto de referencia frente a modelos más complejos.

* ⚙️ **Configuración inicial**. Rutas, paquetes y funciones.
* ♻️ **Carga de datos**.
* 📈 **Evaluación del modelo**.

# ⚙️ Configuración

## Rutas (paths)

In [1]:
import os

# Rutas en función de la carpeta raíz del proyecto (README.md)
base_path = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in locals() else os.path.abspath("..")

# Subrutas
data_dir = os.path.join(base_path, "data")
db_path = os.path.join(data_dir, "steam.db")
src_dir = os.path.join(base_path, "src")
models_dir = os.path.join(base_path, "models")
results_dir = os.path.join(base_path, "res")

## Paquetes y funciones

In [ ]:
# Sistema y configuración
import warnings
warnings.filterwarnings("ignore")
import sqlite3

# Análisis y manipulación de datos
import pandas as pd
import numpy as np

# Funciones personalizadas    
from utils.evaluation_functions import evaluate_popularity_LMO_at_k

# ♻️ Carga de datos

In [ ]:
conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute("""
    CREATE TABLE IF NOT EXISTS games_popular_final AS
    SELECT f.* 
    FROM games_filtered AS f 
    JOIN games_escaled_final AS e 
        ON f.appid = e.appid;
""")

conn.close()

In [ ]:
conn = sqlite3.connect(db_path)
cur = conn.cursor()

games = pd.read_sql("SELECT * FROM games_popular_final", conn)

conn.close()

# Hacemos una copia para trabajar
games = games.copy()

In [6]:
# Tablas test - train
conn = sqlite3.connect(db_path)
cur = conn.cursor()

user_train = pd.read_sql("SELECT * FROM user_train_LMO", sqlite3.connect(db_path))
user_test = pd.read_sql("SELECT * FROM user_test_LMO", sqlite3.connect(db_path))

conn.close()

# Hacemos una copia para trabajar
user_train = user_train.copy()
user_test = user_test.copy()

In [ ]:
# Normalización de tipos
games["appid"] = games["appid"].astype(np.int64)

user_train["user_id"] = user_train["user_id"].astype(np.int64)
user_train["appid"] = user_train["appid"].astype(np.int64)
user_test["user_id"] = user_test["user_id"].astype(np.int64)
user_test["appid"] = user_test["appid"].astype(np.int64)

# 📈 Evaluación del modelo

In [8]:
# Evaluación del modelo base con k = 10
evaluate_popularity_LMO_at_k(
    k=10, 
    games=games, 
    user_train=user_train, 
    user_test=user_test, 
    results_dir=results_dir, 
    tag="20251130")

{'k': 10,
 'precision@k': 0.024194259558653702,
 'recall@k': 0.080647531862179,
 'f1@k': 0.03722193778254416,
 'ndcg@k': 0.1129095797090253,
 'hit_rate@k': 0.21918239653977786,
 'item_coverage@k': 0.05354969574036511,
 'num_users': 47627}

In [9]:
# Evaluación del modelo base con k = 50
evaluate_popularity_LMO_at_k(
    k=50, 
    games=games, 
    user_train=user_train, 
    user_test=user_test, 
    results_dir=results_dir, 
    tag="20251130")

{'k': 50,
 'precision@k': 0.013588510718709977,
 'recall@k': 0.22647517864516623,
 'f1@k': 0.025638699469264108,
 'ndcg@k': 0.17406721243857715,
 'hit_rate@k': 0.5127973628404057,
 'item_coverage@k': 0.1643002028397566,
 'num_users': 47627}